In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, 
    to_date, trim, upper, regexp_replace
)
from pyspark.sql.types import FloatType, IntegerType

# Paths
storage_account = "theportfoliostorage"
container = "datalake"

silver_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/silver/healthcare/"

# Read from Bronze Delta table
df_bronze = spark.table("healthcare.bronze.cms_hrrp_raw")

print(f"Bronze row count: {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
# Define all suppression values in one place - easy to extend later
SUPPRESSED_VALUES = ["Too Few to Report", "N/A", "Not Available", " N/A"]

# Step 1: Add suppression flag BEFORE casting
df_flagged = df_bronze.withColumn(
    "is_suppressed",
    when(
        (col("number_of_discharges").isin(SUPPRESSED_VALUES)) |
        (col("excess_readmission_ratio").isin(SUPPRESSED_VALUES)) |
        (col("number_of_readmissions").isin(SUPPRESSED_VALUES)),
        lit(True)
    ).otherwise(lit(False))
)

# Step 2: Replace ALL suppression values with null
def clean_suppressed(column):
    result = col(column)
    for val in SUPPRESSED_VALUES:
        result = when(col(column) == val, None).otherwise(result)
    return result

df_nulled = df_flagged \
    .withColumn("number_of_discharges", clean_suppressed("number_of_discharges")) \
    .withColumn("excess_readmission_ratio", clean_suppressed("excess_readmission_ratio")) \
    .withColumn("predicted_readmission_rate", clean_suppressed("predicted_readmission_rate")) \
    .withColumn("expected_readmission_rate", clean_suppressed("expected_readmission_rate")) \
    .withColumn("number_of_readmissions", clean_suppressed("number_of_readmissions"))

suppressed_count = df_nulled.filter(col("is_suppressed") == True).count()
unsuppressed_count = df_nulled.filter(col("is_suppressed") == False).count()
print(f"Suppressed rows: {suppressed_count}")
print(f"Unsuppressed rows: {unsuppressed_count}")
print(f"Total: {suppressed_count + unsuppressed_count}")

In [0]:
# Step 3: Cast columns to correct types
df_cast = df_nulled \
    .withColumn("number_of_discharges", 
        col("number_of_discharges").cast(IntegerType())) \
    .withColumn("excess_readmission_ratio", 
        col("excess_readmission_ratio").cast(FloatType())) \
    .withColumn("predicted_readmission_rate", 
        col("predicted_readmission_rate").cast(FloatType())) \
    .withColumn("expected_readmission_rate", 
        col("expected_readmission_rate").cast(FloatType())) \
    .withColumn("number_of_readmissions", 
        col("number_of_readmissions").cast(IntegerType()))

# Step 4: Fix date inconsistency - start_date is string, end_date is already date
df_dates = df_cast \
    .withColumn("start_date", to_date(col("start_date"), "MM/dd/yyyy")) \
    .withColumn("end_date", col("end_date"))

# Step 5: Clean string columns - trim whitespace and standardise state codes
df_cleaned = df_dates \
    .withColumn("facility_name", trim(col("facility_name"))) \
    .withColumn("state", trim(upper(col("state")))) \
    .withColumn("measure_name", trim(col("measure_name")))

# Verify schema looks correct now
df_cleaned.printSchema()

In [0]:
# Step 6: Deduplicate - use facility_id + measure_name as the natural key
# A hospital should only appear once per condition
df_deduped = df_cleaned.dropDuplicates(["facility_id", "measure_name"])

duplicate_count = df_cleaned.count() - df_deduped.count()
print(f"Duplicate rows removed: {duplicate_count}")
print(f"Final row count: {df_deduped.count()}")

# Step 7: Add Silver metadata columns
df_silver = df_deduped \
    .withColumn("_silver_processed_at", current_timestamp()) \
    .withColumn("_silver_version", lit(1))

In [0]:
# Write to Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare.silver.cms_hrrp_clean")

print("Silver table written successfully")

# Verify
spark.sql("""
    SELECT 
        is_suppressed,
        COUNT(*) as row_count,
        AVG(excess_readmission_ratio) as avg_excess_ratio
    FROM healthcare.silver.cms_hrrp_clean
    GROUP BY is_suppressed
""").show()


In [0]:
# Explore all unique non-numeric values in key columns
# to make sure we haven't missed any suppression patterns
from pyspark.sql.functions import regexp_replace

print("Unique non-numeric values in number_of_discharges:")
df_bronze.filter(
    ~col("number_of_discharges").rlike("^[0-9]+$")
).groupBy("number_of_discharges") \
 .count() \
 .orderBy("count", ascending=False) \
 .show(truncate=False)

print("Unique non-numeric values in excess_readmission_ratio:")
df_bronze.filter(
    ~col("excess_readmission_ratio").rlike("^[0-9.]+$")
).groupBy("excess_readmission_ratio") \
 .count() \
 .orderBy("count", ascending=False) \
 .show(truncate=False)